In [1]:
import pandas as pd

In [2]:
df_raw = pd.read_csv('./data/cirrhosis.csv').drop(columns=["ID", "N_Days"])

target_col = 'Status'

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from utils import create_evaluation_dataframe, Metric

In [4]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(exclude=["number"]).columns

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]
y = LabelEncoder().fit_transform(y)

print(f"Kształt danych po usunięciu braków: {X.shape}")

Kształt danych po usunięciu braków: (418, 17)


In [5]:
X_train_cl_tmp, X_test, y_train_cl_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_cl_tmp, y_train_cl_tmp, test_size=0.15/0.85, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

X_train: (292, 17)
X_val: (63, 17)
X_test: (63, 17)


In [6]:
num_pipelines = {
    "KNN imputer + StandardScaler": Pipeline([
        ('knn imputer', KNNImputer()),
        ('standard scaler', StandardScaler())
    ])
}

cat_pipelines = {
    "SimpleImputer + OneHotEncoder": Pipeline([
        ('Unpecified simple imputer', SimpleImputer(strategy='constant', fill_value='Unspecified')),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
}

In [7]:
from sklearn.ensemble import RandomForestClassifier
import itertools

criteria = ['gini', 'entropy']
max_depths = [None, 3, 5, 10, 15]
max_features = ['sqrt', 'log2']
ccp_alphas = [0.0, 0.005, 0.01, 0.02]
bootstrap = [True, False]

models = {}

for crit, depth, feat, alpha, boot in itertools.product(criteria, max_depths, max_features, ccp_alphas, bootstrap):
    model_name = f"paras: {crit}_d{depth}_mf{feat}_a{alpha}_b{boot}"
    models[model_name] = RandomForestClassifier(
        criterion=crit,
        max_depth=depth,
        max_features=feat,
        ccp_alpha=alpha,
        bootstrap=boot,
        random_state=42
    )

# Ewaluacja modeli
results_df = create_evaluation_dataframe(
    X_train,
    y_train,
    X_val,
    y_val,
    num_pipelines,
    cat_pipelines,
    models,
    Metric.F1_SCORE
)

display(results_df)

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: gini_d10_mfsqrt_a0.0_bFalse,0.9897,0.7778,0.9899,0.7480,0.9897,0.7778,0.9897,0.7609
1,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: gini_dNone_mfsqrt_a0.02_bTrue,0.8014,0.7778,0.7517,0.7392,0.8014,0.7778,0.7747,0.7578
2,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: entropy_d5_mfsqrt_a0.02_bTrue,0.8664,0.7778,0.8754,0.7392,0.8664,0.7778,0.8551,0.7578
3,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: gini_dNone_mfsqrt_a0.02_bFalse,0.8014,0.7778,0.7530,0.7380,0.8014,0.7778,0.7738,0.7554
4,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: entropy_d5_mfsqrt_a0.0_bTrue,0.8938,0.7778,0.9040,0.7380,0.8938,0.7778,0.8879,0.7554
...,...,...,...,...,...,...,...,...,...,...,...
155,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: entropy_d10_mflog2_a0.02_bTrue,0.9007,0.6984,0.9067,0.6607,0.9007,0.6984,0.8933,0.6783
156,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: gini_d5_mfsqrt_a0.005_bTrue,0.8733,0.6984,0.8824,0.6577,0.8733,0.6984,0.8598,0.6744
157,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: gini_d5_mfsqrt_a0.0_bTrue,0.8836,0.6984,0.8909,0.6577,0.8836,0.6984,0.8723,0.6744
158,KNN imputer + StandardScaler,SimpleImputer + OneHotEncoder,paras: entropy_d3_mflog2_a0.02_bTrue,0.7842,0.6984,0.7424,0.6562,0.7842,0.6984,0.7564,0.6697


In [ ]:
best_model = RandomForestClassifier(
    criterion='gini',
    max_depth=10,
    max_features='sqrt',
    ccp_alpha=0.0,
    bootstrap=False,
    random_state=42
)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipelines["KNN imputer + StandardScaler"], X_train.select_dtypes(include=["number"]).columns),
        ("cat", cat_pipelines["SimpleImputer + OneHotEncoder"], X_train.select_dtypes(exclude=["number"]).columns),
    ]
)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", best_model)
])

final_pipeline.fit(X_train, y_train)

y_train_pred = final_pipeline.predict(X_train)
y_val_pred = final_pipeline.predict(X_val)
y_test_pred = final_pipeline.predict(X_test)

def print_metrics(y_true, y_pred, set_name):
    print(f"--- {set_name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}\n")

print_metrics(y_train, y_train_pred, "Train")
print_metrics(y_val, y_val_pred, "Validation")
print_metrics(y_test, y_test_pred, "Test")

--- Train ---
Accuracy:  0.8014
Precision: 0.7517
Recall:    0.8014
F1 Score:  0.7747

--- Validation ---
Accuracy:  0.7778
Precision: 0.7392
Recall:    0.7778
F1 Score:  0.7578

--- Test ---
Accuracy:  0.7143
Precision: 0.6718
Recall:    0.7143
F1 Score:  0.6908

